In [1]:
import numpy as np

points = {
    "A": np.array([2, 3]),
    "B": np.array([5, 7]),
    "C": np.array([6, 2]),
    "D": np.array([8, 5]),
    "E": np.array([1, 8])
}

In [2]:
def euclidean_distance(p1, p2):
    return np.sqrt(np.sum((p1 - p2) ** 2))


def manhattan_distance(p1, p2):
    return np.sum(np.abs(p1 - p2))

def mahalanobis_distance(p1, p2, inv_cov_matrix):
    # Difference vector
    diff = p1 - p2

    # Matrix multiplication:
    # diff^T * inv_cov_matrix * diff
    distance_squared = np.dot(np.dot(diff.T, inv_cov_matrix), diff)

    # Square root
    distance = np.sqrt(distance_squared)

    return distance

In [3]:
names = list(points.keys())

euclidean_results = []
manhattan_results = []
mahalanobis_results = []

# Compute covariance matrix using all points
data = np.array(list(points.values()))
cov_matrix = np.cov(data.T)
inv_cov_matrix = np.linalg.inv(cov_matrix)


for i in range(len(names)):
    for j in range(i+1, len(names)):

        p1 = points[names[i]]
        p2 = points[names[j]]

        e = euclidean_distance(p1, p2)
        m = manhattan_distance(p1, p2)
        mh = mahalanobis_distance(p1, p2, inv_cov_matrix)

        euclidean_results.append((names[i], names[j], e))
        manhattan_results.append((names[i], names[j], m))
        mahalanobis_results.append((names[i], names[j], mh))


print("Euclidean Distances")
for r in euclidean_results:
    print(r)

print()

print("Manhattan Distances")
for r in manhattan_results:
    print(r)

print()

print("Mahalanobis Distances")
for r in mahalanobis_results:
    print(r)

print()

print("Shortest Euclidean Distance:")
print(min(euclidean_results, key=lambda x: x[2]))

print()

print("Shortest Manhattan Distance:")
print(min(manhattan_results, key=lambda x: x[2]))

print()

print("Shortest Mahalanobis Distance:")
print(min(mahalanobis_results, key=lambda x: x[2]))

Euclidean Distances
('A', 'B', 5.0)
('A', 'C', 4.123105625617661)
('A', 'D', 6.324555320336759)
('A', 'E', 5.0990195135927845)
('B', 'C', 5.0990195135927845)
('B', 'D', 3.605551275463989)
('B', 'E', 4.123105625617661)
('C', 'D', 3.605551275463989)
('C', 'E', 7.810249675906654)
('D', 'E', 7.615773105863909)

Manhattan Distances
('A', 'B', 7)
('A', 'C', 5)
('A', 'D', 8)
('A', 'E', 6)
('B', 'C', 6)
('B', 'D', 5)
('B', 'E', 5)
('C', 'D', 5)
('C', 'E', 11)
('D', 'E', 10)

Mahalanobis Distances
('A', 'B', 2.2400094970029163)
('A', 'C', 1.3888550806987812)
('A', 'D', 2.563237533477909)
('A', 'E', 1.979180824838224)
('B', 'C', 1.979180824838224)
('B', 'D', 1.1504115302040339)
('B', 'E', 1.3888550806987812)
('C', 'D', 1.6162053280726467)
('C', 'E', 2.5835068799051037)
('D', 'E', 2.471830557561089)

Shortest Euclidean Distance:
('B', 'D', 3.605551275463989)

Shortest Manhattan Distance:
('A', 'C', 5)

Shortest Mahalanobis Distance:
('B', 'D', 1.1504115302040339)


In [4]:
def compare_mahalanobis_and_whitened_euclidean(data, p1, p2):

    mean = np.mean(data, axis=0)
    centered = data - mean

    covariance = np.cov(centered.T)

    inv_covariance = np.linalg.inv(covariance)

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)

    D_inv_sqrt = np.diag(1.0 / np.sqrt(eigenvalues))

    whitening_matrix = eigenvectors @ D_inv_sqrt @ eigenvectors.T   #     # S^{-1/2} = QΛ^{-1/2}Qᵀ

    whitened_data = centered @ whitening_matrix.T   #  z = S^{-1/2}(x-μ)

    idx1 = np.where(np.all(data == p1, axis=1))[0][0]
    idx2 = np.where(np.all(data == p2, axis=1))[0][0]

    wp1 = whitened_data[idx1]
    wp2 = whitened_data[idx2]

    euclidean_whitened = np.sqrt(np.sum((wp1 - wp2) ** 2))

    diff = p1 - p2

    mahalanobis = np.sqrt(diff.T @ inv_covariance @ diff)

    return (euclidean_whitened, mahalanobis, whitened_data)


compare_mahalanobis_and_whitened_euclidean(data, points["A"], points["B"])

(2.240009497002916,
 2.2400094970029163,
 array([[-0.98316252, -0.95925614],
        [ 0.33576108,  0.85129159],
        [ 0.39552704, -1.12698664],
        [ 1.29480288,  0.21592908],
        [-1.04292848,  1.0190221 ]]))